In [1]:
import zipfile
import os

zip_path = "njmin.zip"
extract_path = "njmin_files"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

print(os.listdir(extract_path))

['read.me', 'public.dat', 'survey1.nj', 'survey2.nj', 'check.sas', 'codebook']


In [5]:
import pandas as pd

colspecs = [
    (0, 3),
    (4, 5),
    (6, 7),
    (8, 9),
    (10, 11),
    (12, 13),
    (14, 15),
    (16, 17),
    (18, 19),
    (20, 21),
    (22, 24),
    (25, 30),
    (31, 36),
    (37, 42),
    (43, 48),
    (49, 54),
    (55, 60),
    (61, 62),
    (63, 68),
    (69, 70),
    (71, 76),
    (77, 82),
    (83, 88),
    (89, 94),
    (95, 100),
    (101, 103),
    (104, 106),
    (107, 108),
    (109, 110),
    (111, 117),
    (118, 120),
    (121, 126),
    (127, 132),
    (133, 138),
    (139, 144),
    (145, 150),
    (151, 156),
    (157, 158),
    (159, 160),
    (161, 166),
    (167, 172),
    (173, 178),
    (179, 184),
    (185, 190),
    (191, 193),
    (194, 196),
]

names = [
    "SHEET",
    "CHAIN",
    "CO_OWNED",
    "STATE",
    "SOUTHJ",
    "CENTRALJ",
    "NORTHJ",
    "PA1",
    "PA2",
    "SHORE",
    "NCALLS",
    "EMPFT",
    "EMPPT",
    "NMGRS",
    "WAGE_ST",
    "INCTIME",
    "FIRSTINC",
    "BONUS",
    "PCTAFF",
    "MEALS",
    "OPEN",
    "HRSOPEN",
    "PSODA",
    "PFRY",
    "PENTREE",
    "NREGS",
    "NREGS11",
    "TYPE2",
    "STATUS2",
    "DATE2",
    "NCALLS2",
    "EMPFT2",
    "EMPPT2",
    "NMGRS2",
    "WAGE_ST2",
    "INCTIME2",
    "FIRSTIN2",
    "SPECIAL2",
    "MEALS2",
    "OPEN2R",
    "HRSOPEN2",
    "PSODA2",
    "PFRY2",
    "PENTREE2",
    "NREGS2",
    "NREGS112",
]

df = pd.read_fwf("njmin_files/public.dat", colspecs=colspecs, names=names)

for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["EMPTOT"] = df["EMPPT"] * 0.5 + df["EMPFT"] + df["NMGRS"]
df["EMPTOT2"] = df["EMPPT2"] * 0.5 + df["EMPFT2"] + df["NMGRS2"]
df["DEMP"] = df["EMPTOT2"] - df["EMPTOT"]

df["GAP"] = 0.0
df.loc[(df["STATE"] == 1) & (df["WAGE_ST"] < 5.05) & (df["WAGE_ST"] > 0), "GAP"] = (
    5.05 - df["WAGE_ST"]
) / df["WAGE_ST"]
df.loc[df["WAGE_ST"].isna(), "GAP"] = pd.NA

print(df.shape)
print(
    df[
        [
            "SHEET",
            "STATE",
            "WAGE_ST",
            "WAGE_ST2",
            "EMPTOT",
            "EMPTOT2",
            "DEMP",
            "GAP",
            "STATUS2",
        ]
    ].head()
)


(410, 50)
   SHEET  STATE  WAGE_ST  WAGE_ST2  EMPTOT  EMPTOT2   DEMP  GAP  STATUS2
0     46      0      NaN      4.30   40.50     24.0 -16.50  NaN        1
1     49      0      NaN      4.45   13.75     11.5  -2.25  NaN        1
2    506      0      NaN      5.00    8.50     10.5   2.00  NaN        1
3     56      0      5.0      5.25   34.00     20.0 -14.00  0.0        1
4     61      0      5.5      4.75   24.00     35.5  11.50  0.0        1


In [6]:
print(df.shape)
print(
    df[
        [
            "SHEET",
            "STATE",
            "WAGE_ST",
            "WAGE_ST2",
            "EMPTOT",
            "EMPTOT2",
            "DEMP",
            "GAP",
            "STATUS2",
        ]
    ].head()
)


(410, 50)
   SHEET  STATE  WAGE_ST  WAGE_ST2  EMPTOT  EMPTOT2   DEMP  GAP  STATUS2
0     46      0      NaN      4.30   40.50     24.0 -16.50  NaN        1
1     49      0      NaN      4.45   13.75     11.5  -2.25  NaN        1
2    506      0      NaN      5.00    8.50     10.5   2.00  NaN        1
3     56      0      5.0      5.25   34.00     20.0 -14.00  0.0        1
4     61      0      5.5      4.75   24.00     35.5  11.50  0.0        1


In [7]:
print(df["STATE"].value_counts(dropna=False).sort_index())
print(df[["EMPTOT", "EMPTOT2", "WAGE_ST", "WAGE_ST2", "STATUS2"]].isna().sum())


STATE
0     79
1    331
Name: count, dtype: int64
EMPTOT      12
EMPTOT2     14
WAGE_ST     20
WAGE_ST2    21
STATUS2      0
dtype: int64


In [8]:
state_names = {0: "PA", 1: "NJ"}

basic_check = (
    df.groupby("STATE")[["EMPTOT", "WAGE_ST", "WAGE_ST2"]]
    .mean()
    .rename(index=state_names)
)

print(basic_check)


          EMPTOT   WAGE_ST  WAGE_ST2
STATE                               
PA     23.331169  4.630132  4.617465
NJ     20.439408  4.612134  5.080849


In [9]:
import pandas as pd

df["STATE_NAME"] = df["STATE"].map({0: "PA", 1: "NJ"})

df["WAGE_GROUP"] = pd.NA
df.loc[(df["STATE"] == 1) & (df["WAGE_ST"] == 4.25), "WAGE_GROUP"] = "4.25"
df.loc[
    (df["STATE"] == 1) & (df["WAGE_ST"] >= 4.26) & (df["WAGE_ST"] <= 4.99), "WAGE_GROUP"
] = "4.26-4.99"
df.loc[(df["STATE"] == 1) & (df["WAGE_ST"] >= 5.00), "WAGE_GROUP"] = "5.00+"

row1 = df.groupby("STATE_NAME")["EMPTOT"].mean()
row2 = df.groupby("STATE_NAME")["EMPTOT2"].mean()
row3 = row2 - row1

balanced = df[df["EMPTOT"].notna() & df["EMPTOT2"].notna()].copy()
row4 = balanced.groupby("STATE_NAME")["DEMP"].mean()

temp_closed_codes = [2, 3]
balanced_zero = balanced.copy()
balanced_zero.loc[balanced_zero["STATUS2"].isin(temp_closed_codes), "EMPTOT2"] = 0
balanced_zero["DEMP_zero"] = balanced_zero["EMPTOT2"] - balanced_zero["EMPTOT"]
row5 = balanced_zero.groupby("STATE_NAME")["DEMP_zero"].mean()

nj = df[df["STATE"] == 1].copy()
row1_nj = nj.groupby("WAGE_GROUP")["EMPTOT"].mean()
row2_nj = nj.groupby("WAGE_GROUP")["EMPTOT2"].mean()
row3_nj = row2_nj - row1_nj

balanced_nj = nj[nj["EMPTOT"].notna() & nj["EMPTOT2"].notna()].copy()
row4_nj = balanced_nj.groupby("WAGE_GROUP")["DEMP"].mean()

balanced_nj_zero = balanced_nj.copy()
balanced_nj_zero.loc[balanced_nj_zero["STATUS2"].isin(temp_closed_codes), "EMPTOT2"] = 0
balanced_nj_zero["DEMP_zero"] = balanced_nj_zero["EMPTOT2"] - balanced_nj_zero["EMPTOT"]
row5_nj = balanced_nj_zero.groupby("WAGE_GROUP")["DEMP_zero"].mean()

table3 = pd.DataFrame(
    {
        "PA": [row1["PA"], row2["PA"], row3["PA"], row4["PA"], row5["PA"]],
        "NJ": [row1["NJ"], row2["NJ"], row3["NJ"], row4["NJ"], row5["NJ"]],
        "4.25": [
            row1_nj["4.25"],
            row2_nj["4.25"],
            row3_nj["4.25"],
            row4_nj["4.25"],
            row5_nj["4.25"],
        ],
        "4.26-4.99": [
            row1_nj["4.26-4.99"],
            row2_nj["4.26-4.99"],
            row3_nj["4.26-4.99"],
            row4_nj["4.26-4.99"],
            row5_nj["4.26-4.99"],
        ],
        "5.00+": [
            row1_nj["5.00+"],
            row2_nj["5.00+"],
            row3_nj["5.00+"],
            row4_nj["5.00+"],
            row5_nj["5.00+"],
        ],
    },
    index=[
        "1. FTE employment before, all available observations",
        "2. FTE employment after, all available observations",
        "3. Change in mean FTE employment",
        "4. Change in mean FTE employment, balanced sample",
        "5. Change in mean FTE employment, temp closed set to 0",
    ],
)

table3["NJ-PA"] = table3["NJ"] - table3["PA"]
table3["Low-High"] = table3["4.25"] - table3["5.00+"]
table3["Mid-High"] = table3["4.26-4.99"] - table3["5.00+"]

print(table3.round(2))


                                                       PA     NJ   4.25  \
1. FTE employment before, all available observa...  23.33  20.44  19.56   
2. FTE employment after, all available observat...  21.17  21.03  20.88   
3. Change in mean FTE employment                    -2.17   0.59   1.32   
4. Change in mean FTE employment, balanced sample   -2.28   0.47   1.20   
5. Change in mean FTE employment, temp closed s...  -2.28   0.47   1.20   

                                                    4.26-4.99  5.00+  NJ-PA  \
1. FTE employment before, all available observa...      20.08  22.25  -2.89   
2. FTE employment after, all available observat...      20.96  20.21  -0.14   
3. Change in mean FTE employment                         0.87  -2.04   2.75   
4. Change in mean FTE employment, balanced sample        0.71  -2.16   2.75   
5. Change in mean FTE employment, temp closed s...       0.71  -2.16   2.75   

                                                    Low-High  Mid-High  
1

In [10]:
print(df["STATUS2"].value_counts(dropna=False).sort_index())


STATUS2
0      1
1    399
2      2
3      6
4      1
5      1
Name: count, dtype: int64


In [11]:
print(df.groupby("STATUS2")[["EMPTOT2", "STATE"]].agg(["count", "mean"]))


        EMPTOT2            STATE          
          count       mean count      mean
STATUS2                                   
0             0        NaN     1  1.000000
1           390  21.378205   399  0.804511
2             0        NaN     2  1.000000
3             6   0.000000     6  0.833333
4             0        NaN     1  1.000000
5             0        NaN     1  1.000000


In [14]:
row5_sample = df[df["EMPTOT"].notna() & df["STATUS2"].notna()].copy()
row5_sample["EMPTOT2_row5"] = row5_sample["EMPTOT2"]

row5_sample.loc[row5_sample["STATUS2"].isin([4, 5]), "EMPTOT2_row5"] = 0

row5_sample["DEMP_row5"] = row5_sample["EMPTOT2_row5"] - row5_sample["EMPTOT"]
row5 = row5_sample.groupby("STATE_NAME")["DEMP_row5"].mean()

row5_nj_sample = row5_sample[row5_sample["STATE"] == 1].copy()
row5_nj = row5_nj_sample.groupby("WAGE_GROUP")["DEMP_row5"].mean()

table3.loc["5. Change in mean FTE employment, temp closed set to 0", "PA"] = row5["PA"]
table3.loc["5. Change in mean FTE employment, temp closed set to 0", "NJ"] = row5["NJ"]
table3.loc["5. Change in mean FTE employment, temp closed set to 0", "4.25"] = row5_nj[
    "4.25"
]
table3.loc["5. Change in mean FTE employment, temp closed set to 0", "4.26-4.99"] = (
    row5_nj["4.26-4.99"]
)
table3.loc["5. Change in mean FTE employment, temp closed set to 0", "5.00+"] = row5_nj[
    "5.00+"
]

table3["NJ-PA"] = table3["NJ"] - table3["PA"]
table3["Low-High"] = table3["4.25"] - table3["5.00+"]
table3["Mid-High"] = table3["4.26-4.99"] - table3["5.00+"]

print(table3.round(2))


                                                       PA     NJ   4.25  \
1. FTE employment before, all available observa...  23.33  20.44  19.56   
2. FTE employment after, all available observat...  21.17  21.03  20.88   
3. Change in mean FTE employment                    -2.17   0.59   1.32   
4. Change in mean FTE employment, balanced sample   -2.28   0.47   1.20   
5. Change in mean FTE employment, temp closed s...  -2.28   0.36   1.20   

                                                    4.26-4.99  5.00+  NJ-PA  \
1. FTE employment before, all available observa...      20.08  22.25  -2.89   
2. FTE employment after, all available observat...      20.96  20.21  -0.14   
3. Change in mean FTE employment                         0.87  -2.04   2.75   
4. Change in mean FTE employment, balanced sample        0.71  -2.16   2.75   
5. Change in mean FTE employment, temp closed s...       0.60  -2.39   2.65   

                                                    Low-High  Mid-High  
1

## Replicate Table 4

In [15]:
import statsmodels.api as sm

reg1 = df[
    df["EMPTOT"].notna()
    & df["EMPTOT2"].notna()
    & df["WAGE_ST"].notna()
    & df["WAGE_ST2"].notna()
].copy()

X = sm.add_constant(reg1["STATE"])
y = reg1["DEMP"]

model1 = sm.OLS(y, X).fit()

print(model1.summary())


                            OLS Regression Results                            
Dep. Variable:                   DEMP   R-squared:                       0.010
Model:                            OLS   Adj. R-squared:                  0.008
Method:                 Least Squares   F-statistic:                     3.658
Date:                Tue, 21 Apr 2026   Prob (F-statistic):             0.0566
Time:                        11:49:52   Log-Likelihood:                -1257.0
No. Observations:                 351   AIC:                             2518.
Df Residuals:                     349   BIC:                             2526.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -1.8788      1.073     -1.751      0.0